In [1]:
from src.evaluation.evaluacion import (evaluar_interna,
                                       restaurar_variable_clase,
                                       evaluar_externa)
import pandas as pd
import logging
import os

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Cargar el archivo exportado
ruta_clusterizado = "../data/processed/vehiculos_clusterizado.csv"
df_clusterizado = pd.read_csv(ruta_clusterizado)

# Configurar ruta de archivo
ruta_clusterizado = "../data/processed/vehiculos_clusterizado.csv"

# Cargar el archivo CSV
df_clusterizado = pd.read_csv(ruta_clusterizado)
logging.info(f"Data clusterizada cargada desde: {ruta_clusterizado}")

# Verificar columnas
logging.info(f"Columnas disponibles: {list(df_clusterizado.columns)}")

# Directorio de salida
ruta_evaluacion = "../outputs/04_evaluacion"

2025-07-02 19:10:48,949 - INFO - Data clusterizada cargada desde: ../data/processed/vehiculos_clusterizado.csv
2025-07-02 19:10:48,950 - INFO - Columnas disponibles: ['year', 'desplazamiento', 'cilindros', 'co2', 'consumo_litros_milla', 'clase_tipo_Coche Familiar', 'clase_tipo_Coches Grandes', 'clase_tipo_Coches Medianos', 'clase_tipo_Coches pequeños', 'clase_tipo_Deportivos', 'clase_tipo_Furgoneta', 'clase_tipo_Vehículos Especiales', 'traccion_tipo_dos', 'transmision_tipo_Manual', 'combustible_tipo_Otros tipos de combustible', 'combustible_tipo_Premium', 'tamano_motor_tipo_mediano', 'tamano_motor_tipo_muy grande', 'tamano_motor_tipo_muy pequeño', 'tamano_motor_tipo_pequeño', 'consumo_tipo_bajo', 'consumo_tipo_moderado', 'consumo_tipo_muy alto', 'consumo_tipo_muy bajo', 'co2_tipo_bajo', 'co2_tipo_moderado', 'co2_tipo_muy alto', 'co2_tipo_muy bajo', 'cluster']


In [2]:
# 1. Evaluación interna
resultados_interna = evaluar_interna(df_clusterizado, save_path=ruta_evaluacion)

2025-07-02 19:11:13,451 - INFO - Evaluación interna completada y exportada


Resultados de evaluación interna:
1. Coeficiente de Silhouette = 0.2352
- Clustering medianamente compacto, con cierto solapamiento

2. Davied-Bouldin Index (DBI) = 1.5523
- Separación moderada, con cierta superposición

3. Calinski-Harabasz Indes (CHI) = 8589.6585
- Valor alto que confirma que la varianza entre clusters es mucho mayor que la interna

In [3]:
# Cargar data original antes de dummificación
df_original = pd.read_csv("../data/raw/vehiculos.csv")

# Reconstruir columna
df_clusterizado = restaurar_variable_clase(df_clusterizado, df_original)

2025-07-02 19:11:13,663 - INFO - Clase faltante detectada: 'Camionetas'
2025-07-02 19:11:13,726 - INFO - Variable 'clase_tipo' restaurada correctamente en el DataFrame


In [4]:
# 2. Evaluación externa (con 8 clusters y etiqueta de referencia)
resultados_externa = evaluar_externa(
    df=df_clusterizado,
    etiqueta_referencia="clase_tipo",
    save_path=ruta_evaluacion
)

2025-07-02 19:11:14,265 - INFO - Evaluación externa completada y exportada a: ../outputs/04_evaluacion\evaluacion_externa.html


Resultados evaluación externa:

1. Adjusted Rand Index (ARI)
- Medición: Coincidencia entre etiquetas reales y predichas, ajustando por el azar.
- Rango de valores: -1 a 1 (0 indica coincidencia aleatoria).
- Rango ideal: > 0.6 (excelente si > 0.8).
- Evaluación: El valor 0.0594 indica poca relación entre los clusters encontrados y las verdaderas clases (clase_tipo).

2. Normalized Mutual Information (NMI)
- Medición: Cuánta información tienen en común los clusters y las clases reales.
- Rango de valores: 0 a 1.
- Rango ideal: > 0.5.
- Evaluación: Con un valor de 0.1476, la dependencia entre clusters y clases es débil.

3. Homogeneity
- Medición: Si cada cluster contiene sólo muestras de una misma clase.
- Rango de valores: 0 a 1.
- Rango ideal: > 0.6.
- Evaluación: Un valor de 0.1639 sugiere que los clusters contienen muestras de diferentes clases, lo cual no es deseable.

4. Completeness
- Medición: Si todas las muestras de una clase se agrupan en un solo cluster.
- Rango de valores: 0 a 1.
- Rango ideal: > 0.6.
- Evaluación: El valor de 0.1342 indica que una misma clase está repartida entre varios clusters.

5. V-measure
- Medición: Promedio armónico entre homogeneidad y completitud.
- Rango de valores: 0 a 1.
- Rango ideal: > 0.6.
- Evaluación: Con 0.1476, se confirma una baja calidad del agrupamiento respecto a las etiquetas verdaderas.